In [1]:
%pip install pandas requests astropy tqdm
%pip install pandas requests tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Download the next 100 light curves of ngoodobs>2

In [2]:

from pathlib import Path
from io import StringIO
import time
import pandas as pd
import requests
from tqdm.auto import tqdm

# ==========================================
# 1. SETTINGS
# ==========================================

folder = Path(
    r"C:\Users\marya\PycharmProjects\PythonProject"
)

output = folder / "ZTF_100_lightcurves"
output.mkdir(parents=True, exist_ok=True)

RA = 94.3
DEC = -7.8
RADIUS = 0.1666

START_ROW = 101
TARGET = 100
MIN_GOOD_OBS = 3

TAP_URL = "https://irsa.ipac.caltech.edu/TAP/sync"
LC_URL = (
    "https://irsa.ipac.caltech.edu"
    "/cgi-bin/ZTF/nph_light_curves"
)

session = requests.Session()
session.headers["User-Agent"] = "ZTF-research/1.0"


import xml.etree.ElementTree as ET

def tap_query(adql):

    response = session.get(
        TAP_URL,
        params={
            "REQUEST": "doQuery",
            "LANG": "ADQL",
            "FORMAT": "csv",
            "QUERY": adql
        },
        timeout=180
    )

    response.raise_for_status()

    content = response.text.strip()

    # IRSA may return an XML error document
    if content.startswith("<"):

        try:
            root = ET.fromstring(content)

            errors = [
                element.attrib.get("value", "")
                for element in root.iter()
                if element.tag.endswith("INFO")
                and element.attrib.get("name") == "QUERY_STATUS"
            ]

            messages = [
                element.text.strip()
                for element in root.iter()
                if element.tag.endswith("INFO")
                and element.attrib.get("name") == "QUERY_STATUS"
                and element.text
            ]

            raise RuntimeError(
                "IRSA TAP response: "
                + " | ".join(errors + messages)
            )

        except ET.ParseError:
            raise RuntimeError(
                "Unexpected server response: "
                + content[:500]
            )

    return pd.read_csv(
        StringIO(content),
        dtype={"oid": "string"}
    )

# ==========================================
# 2. FIND THE DR24 OBJECTS TABLE
# ==========================================

print("Finding ZTF DR24 catalog at IRSA...")

tables = tap_query("""
SELECT table_name
FROM TAP_SCHEMA.tables
WHERE LOWER(table_name) LIKE '%ztf%'
""")

names = tables["table_name"].astype(str).tolist()

matches = [
    name for name in names
    if "dr24" in name.lower()
    and "object" in name.lower()
]

if len(matches) != 1:
    raise RuntimeError(
        "Could not uniquely identify DR24 Objects table. "
        f"Candidates: {matches}"
    )

table_name = matches[0]
print("Catalog:", table_name)

# ==========================================
# 3. GET SOURCES IN YOUR SKY REGION
# ==========================================

print("Retrieving ZTF objects...")


query = f"""
SELECT oid, ra, dec, filtercode, ngoodobs
FROM {table_name}
WHERE CONTAINS(
    POINT(ra, dec),
    CIRCLE({RA}, {DEC}, {RADIUS})
) = 1
ORDER BY ra, dec, oid
"""

catalog = tap_query(query)

print("Total sources retrieved:", len(catalog))
print(catalog.head())


catalog = tap_query(query)

catalog["ngoodobs"] = pd.to_numeric(
    catalog["ngoodobs"], errors="coerce"
)

print("Total catalog rows:", len(catalog))

# Save automatically for future use
catalog.to_csv(
    output / "retrieved_ZTF_catalog.csv",
    index=False
)

# ==========================================
# 4. START FROM ROW 101
# ==========================================

candidates = catalog.iloc[START_ROW - 1:].copy()

candidates = candidates[
    candidates["ngoodobs"] >= MIN_GOOD_OBS
]

print("Qualifying rows:", len(candidates))

# ==========================================
# 5. DOWNLOAD LIGHT CURVES
# ==========================================

success = []
failures = []

for index, source in tqdm(
    candidates.iterrows(),
    total=len(candidates)
):

    if len(success) >= TARGET:
        break

    oid = str(source["oid"]).strip()

    if not oid.isdigit():
        failures.append({
            "row": index + 1,
            "oid": oid,
            "error": "Invalid OID"
        })
        continue

    file_path = output / f"ZTF_{oid}.csv"

    try:
        if file_path.exists():
            lc = pd.read_csv(file_path)
        else:
            response = session.get(
                LC_URL,
                params={
                    "ID": oid,
                    "COLLECTION": "ztf_dr24",
                    "FORMAT": "CSV"
                },
                timeout=45
            )

            response.raise_for_status()

            if response.text.lstrip().startswith("<"):
                raise ValueError(
                    "Unexpected HTML/XML response"
                )

            lc = pd.read_csv(
                StringIO(response.text),
                comment="\\"
            )

        if lc.empty:
            raise ValueError("Empty light curve")

        # Check for a usable photometry column
        if not {"mag", "magerr"}.issubset(lc.columns):
            raise ValueError(
                f"Unexpected columns: {lc.columns.tolist()}"
            )

        lc.to_csv(file_path, index=False)

        success.append({
            "catalog_row": index + 1,
            "oid": oid,
            "ra": source["ra"],
            "dec": source["dec"],
            "filtercode": source["filtercode"],
            "ngoodobs": source["ngoodobs"],
            "n_points": len(lc),
            "filename": file_path.name
        })

        print(
            f"{len(success)}/{TARGET} "
            f"Downloaded OID {oid}"
        )

    except Exception as error:
        failures.append({
            "row": index + 1,
            "oid": oid,
            "error": str(error)
        })

        print(f"Failed {oid}: {error}")

    # Be considerate of IRSA servers
    time.sleep(0.5)

# ==========================================
# 6. SAVE RESULTS
# ==========================================

pd.DataFrame(success).to_csv(
    output / "download_summary.csv",
    index=False
)

pd.DataFrame(failures).to_csv(
    output / "failed_downloads.csv",
    index=False
)

print("\nDONE")
print("Downloaded:", len(success))
print("Failed:", len(failures))
print("Saved in:", output)


Finding ZTF DR24 catalog at IRSA...
Catalog: ztf_objects_dr24
Retrieving ZTF objects...
Total sources retrieved: 13669
                oid         ra       dec filtercode  ngoodobs
0   358216400034468  94.133071 -7.806190         zr         1
1  1402105300027500  94.133350 -7.796341         zg         0
2  1402105300008817  94.133409 -7.817906         zg        35
3   358216400019647  94.133410 -7.817910         zr       287
4   358116400018195  94.133413 -7.817930         zg       182
Total catalog rows: 13669
Qualifying rows: 7559


  0%|          | 0/7559 [00:00<?, ?it/s]

1/100 Downloaded OID 358216400018098
2/100 Downloaded OID 358116400012991
3/100 Downloaded OID 1402205300036129
4/100 Downloaded OID 1402205300029150
5/100 Downloaded OID 358216400019267
6/100 Downloaded OID 358216400019438
7/100 Downloaded OID 358116400012202
8/100 Downloaded OID 358216400018583
9/100 Downloaded OID 1402205300011183
10/100 Downloaded OID 1402105300019940
11/100 Downloaded OID 1402205300011048
12/100 Downloaded OID 358216400018423
13/100 Downloaded OID 1402105300015623
14/100 Downloaded OID 1402205300021172
15/100 Downloaded OID 358216400019384
16/100 Downloaded OID 358116400012099
17/100 Downloaded OID 358116400029414
18/100 Downloaded OID 1402205300020675
19/100 Downloaded OID 358216400017944
20/100 Downloaded OID 1402205300012582
21/100 Downloaded OID 1402105300009237
22/100 Downloaded OID 358116400012421
23/100 Downloaded OID 358216400018934
24/100 Downloaded OID 1402205300021026
25/100 Downloaded OID 1402105300008405
26/100 Downloaded OID 358116400021575
27/100 Do

In [3]:
%pip install pandas numpy scipy

Note: you may need to restart the kernel to use updated packages.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

from scipy.stats import skew, kurtosis, linregress

# ==========================================
# 1. FILE LOCATIONS
# ==========================================

folder = Path(
    r"C:\Users\marya\PycharmProjects\PythonProject"
)

input_folder = folder / "ZTF_100_lightcurves"
output_folder = folder / "ztf_results_100"

output_folder.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. FEATURE FUNCTIONS
# ==========================================

def calculate_features(df):

    result = {}

    mag = df["mag"].to_numpy(dtype=float)
    err = df["magerr"].to_numpy(dtype=float)
    time = df["mjd"].to_numpy(dtype=float)

    valid = (
        np.isfinite(mag)
        & np.isfinite(err)
        & np.isfinite(time)
        & (err > 0)
    )

    mag = mag[valid]
    err = err[valid]
    time = time[valid]

    n = len(mag)

    result["n_obs"] = n

    feature_names = [
        "mean", "Fvar", "Fvar_error",
        "max_amplitude", "min_max_diff_by_2",
        "skewness", "kurtosis",
        "linear_trend_mag_per_day",
        "von_neumann_ratio"
    ]

    for name in feature_names:
        result[name] = np.nan

    if n == 0:
        return result

    # Mean magnitude
    result["mean"] = np.mean(mag)

    # Maximum amplitude
    result["max_amplitude"] = (
        np.max(mag) - np.min(mag)
    )

    # Half magnitude range
    result["min_max_diff_by_2"] = (
        np.max(mag) - np.min(mag)
    ) / 2

    if n < 2:
        return result

    # Fractional variability in flux space
    # Convert magnitudes to relative flux
    flux = 10 ** (-0.4 * (mag - np.median(mag)))

    flux_err = (
        np.log(10) / 2.5
    ) * flux * err

    mean_flux = np.mean(flux)

    variance = np.var(flux, ddof=1)
    mean_err2 = np.mean(flux_err ** 2)

    excess_variance = variance - mean_err2

    if excess_variance > 0 and mean_flux > 0:

        fvar = np.sqrt(excess_variance) / mean_flux

        result["Fvar"] = fvar

        # Approximate uncertainty
        n_eff = len(flux)

        normalized_excess = (
            excess_variance / mean_flux ** 2
        )

        err_nxs = np.sqrt(
            (np.sqrt(2 / n_eff) *
             mean_err2 / mean_flux ** 2) ** 2
            +
            (np.sqrt(mean_err2 / n_eff) *
             2 * fvar / mean_flux) ** 2
        )

        result["Fvar_error"] = (
            err_nxs / (2 * fvar)
            if fvar > 0 else np.nan
        )

    # Skewness
    if n >= 3 and np.std(mag) > 0:
        result["skewness"] = skew(
            mag, bias=False
        )

    # Kurtosis (excess kurtosis)
    if n >= 4 and np.std(mag) > 0:
        result["kurtosis"] = kurtosis(
            mag, fisher=True, bias=False
        )

    # Linear trend
    if np.ptp(time) > 0:
        slope, _, _, _, _ = linregress(
            time, mag
        )
        result["linear_trend_mag_per_day"] = slope

    # Von Neumann ratio
    if variance > 0:
        order = np.argsort(time)
        sorted_mag = mag[order]

        mag_variance = np.var(
            sorted_mag, ddof=1
        )

        if mag_variance > 0:
            result["von_neumann_ratio"] = (
                np.sum(np.diff(sorted_mag) ** 2)
                / ((n - 1) * mag_variance)
            )

    return result


# ==========================================
# 3. CLEANING FUNCTION
# ==========================================

def clean_lightcurve(df, sigma=3):

    cleaned = df.copy()

    for column in ["mjd", "mag", "magerr"]:
        cleaned[column] = pd.to_numeric(
            cleaned[column], errors="coerce"
        )

    cleaned = cleaned.dropna(
        subset=["mjd", "mag", "magerr"]
    )

    cleaned = cleaned[
        cleaned["magerr"] > 0
    ]

    # Robust sigma clipping using median and MAD
    if len(cleaned) >= 5:

        mag = cleaned["mag"].to_numpy()

        median = np.median(mag)

        mad = np.median(
            np.abs(mag - median)
        )

        robust_sigma = 1.4826 * mad

        if robust_sigma > 0:

            keep = (
                np.abs(mag - median)
                <= sigma * robust_sigma
            )

            cleaned = cleaned.loc[keep]

    return cleaned


# ==========================================
# 4. PROCESS ALL DOWNLOADED FILES
# ==========================================

rows = []

files = sorted(input_folder.glob("ZTF_*.csv"))

print("Light curves found:", len(files))

for file in files:

    try:
        df = pd.read_csv(file)

        # Check required columns
        required = {"mjd", "mag", "magerr"}

        if not required.issubset(df.columns):
            print("Skipping:", file.name)
            continue

        oid = file.stem.replace("ZTF_", "")

        # Before cleaning
        before = calculate_features(df)

        # After cleaning
        cleaned = clean_lightcurve(df)

        after = calculate_features(cleaned)

        row = {
            "oid": oid,
            "filtercode": (
                df["filtercode"].iloc[0]
                if "filtercode" in df.columns
                else np.nan
            )
        }

        for key, value in before.items():
            row[f"{key}_before"] = value

        for key, value in after.items():
            row[f"{key}_after"] = value

        row["n_removed_before"] = 0
        row["n_removed_after"] = (
            before["n_obs"] - after["n_obs"]
        )

        rows.append(row)

        print("Processed:", oid)

    except Exception as e:
        print("Error:", file.name, e)


# ==========================================
# 5. SAVE FEATURES CATALOGUE
# ==========================================

catalog = pd.DataFrame(rows)

output_path = (
    output_folder / "ztf_features_100.csv"
)

catalog.to_csv(output_path, index=False)

print("\nDONE!")
print("Sources processed:", len(catalog))
print("Saved catalogue:", output_path)

print(catalog.head())


Light curves found: 100
Processed: 1402105300007758
Processed: 1402105300007989
Processed: 1402105300008260
Processed: 1402105300008405
Processed: 1402105300008875
Processed: 1402105300008948
Processed: 1402105300009050
Processed: 1402105300009121
Processed: 1402105300009139
Processed: 1402105300009237
Processed: 1402105300009250
Processed: 1402105300009291
Processed: 1402105300015623
Processed: 1402105300015711
Processed: 1402105300015929
Processed: 1402105300019940
Processed: 1402105300019979
Processed: 1402105300019980
Processed: 1402105300024689
Processed: 1402105300027251
Processed: 1402205300010591
Processed: 1402205300010791
Processed: 1402205300010911
Processed: 1402205300011048
Processed: 1402205300011138
Processed: 1402205300011183
Processed: 1402205300011235
Processed: 1402205300011373
Processed: 1402205300011657
Processed: 1402205300011977
Processed: 1402205300012098
Processed: 1402205300012205
Processed: 1402205300012332
Processed: 1402205300012434
Processed: 1402205300012